========================================================= <br>
Spark Certification Study Notebook Bronze Layer - Data Ingestion and Initial Processing
========================================================= <br>

# Import Environment

In [0]:
# Import environment variables
from setup.env import *

from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions", DEFAULT_SHUFFLE_PARTITIONS)

# Silver Layer - Data Cleaning and Transformations

<b> Silver Layer: Data Cleaning and Business Transformations
Goal </b>

The Silver layer is responsible for transforming raw Bronze data into clean, standardized, and enriched datasets.

This layer performs the core data engineering transformations, including:
- data cleaning
- deduplication
- null handling
- schema standardization
- joins
- nested data processing
- aggregations
- window functions

Silver tables provide trusted datasets that can be safely used by downstream analytical workloads.

Input Tables
The Silver layer reads data from Bronze tables:

| Table              | Description                       |
| ------------------ | --------------------------------- |
| bronze.customers   | Raw customers dataset             |
| bronze.products    | Products dataset with nested JSON |
| bronze.orders      | Orders dataset                    |
| bronze.order_items | Order items dataset               |



Output Tables

The Silver layer will produce the following cleaned datasets:
- spark_cert_catalog.silver.customers_clean
- spark_cert_catalog.silver.products_clean
- spark_cert_catalog.silver.orders_enriched
- spark_cert_catalog.silver.order_items_enriched

## TASK 1 — Load Bronze Tables

<b> Why <b>

Before applying transformations, we must load the Bronze tables into Spark DataFrames.

Using tables instead of files ensures:
- schema consistency
- governance
- reproducibility
- easier downstream processing

This step also reinforces the difference between:
- Spark SQL
- PySpark DataFrame API
- Both are heavily tested in the certification.

<b> Key Concepts </b>
- spark.table()
-DataFrame creation
- Spark SQL queries

### Your Solution

### Suggested Solution

In [0]:
print("=" * 50)
print("Starting Silver Layer Transformations")

In [0]:
customers_bronze_df = spark.table("spark_cert_catalog.bronze.customers")
products_bronze_df = spark.table("spark_cert_catalog.bronze.products")
orders_bronze_df = spark.table("spark_cert_catalog.bronze.orders")
order_items_bronze_df = spark.table("spark_cert_catalog.bronze.order_items")

display(customers_bronze_df.limit(5))

## TASK 2 — Data Quality Inspection

<b> Why </b>

Before cleaning data, must inspect the dataset. </br>
This helps detect:
- null values
- schema inconsistencies
- duplicates
- unexpected distributions

Spark provides several useful methods:
- printSchema()
- describe()
- summary()

<b> Your Task </b></br>
Inspect the customers Bronze DataFrame using:
- printSchema()
- describe()

### Your Solution

### Sugested Solution

In [0]:
customers_bronze_df.printSchema()
customers_bronze_df.describe().show()

In [0]:
products_bronze_df.printSchema()
products_bronze_df.describe()

In [0]:
orders_bronze_df.printSchema()
orders_bronze_df.describe()

In [0]:
order_items_bronze_df.printSchema()
order_items_bronze_df.describe()

## TASK 3 — Handle Missing Data

<b> Why </b></br>
Real-world datasets frequently contain missing values. </br>
Spark provides multiple ways to handle nulls:

| Function     | Purpose                            |
| ------------ | ---------------------------------- |
| `dropna()`   | remove rows containing null values |
| `fillna()`   | replace null values                |
| `nvl()`      | SQL null replacement               |
| `ifnull()`   | SQL null replacement (same nvl())  |
| `coalesce()` | return first non-null value        |

<b> Your Task<b>

Perform missing data handling across all Bronze DataFrames.

1 - Customers </br>
Apply the following logic:
- Remove rows where customer_id is null
- Replace null country values with "unknown"
- Attempt to replace null age values using:
```
fillna({"age": "missing"})
```
Observe that nothing happens, because age is an integer column.

2 - Products </br>
- Replace null price values with 0
- Replace null category values with "unknown"

3 - Orders
- Remove rows where order_id is null
- Replace missing status values with "pending"

4 - Order Items
- Replace null quantity values with 1
- Replace null price values with 0


In [0]:
## Create a fake null line in age
customers_bronze_df = customers_bronze_df.unionByName(
    customers_bronze_df.limit(1).withColumn("age", lit(None))
)

### Your solution

### Sugested solution

In [0]:
from pyspark.sql.functions import col, when, coalesce

customers_silver_df = (
    customers_bronze_df
    # drop rows where primary key is missing
    .dropna(subset=["customer_id"])
    
    # fill missing country values
    .fillna({"country": "unknown"})
    
    # IMPORTANT BEHAVIOR
    # age is an integer column
    # fillna with a string value will NOT modify the column
    # Spark silently ignores incompatible types
    .fillna({"age": "missing"})
)

# Example of using coalesce() to ensure fallback values
customers_silver_df = customers_silver_df.withColumn(
    "country",
    coalesce(col("country"), col("country"))
)


products_silver_df = (
    products_bronze_df
    .fillna({
        "price": 0,
        "category": "unknown",
        "name": "unknown"
    })
)

orders_silver_df = (
    orders_bronze_df
    .dropna(subset=["order_id"])
    .fillna({
        "status": "pending"
    })
)

order_items_silver_df = (
    order_items_bronze_df
    .fillna({
        "quantity": 1,
        "price": 0
    })
)

display(customers_silver_df.filter(col("age").isNull()).limit(5))
display(customers_silver_df.limit(5))
# display(products_silver_df.limit(5))
# display(orders_silver_df.limit(5))
# display(order_items_silver_df.limit(5))

## TASK 4 — Remove Duplicate Records

<b> Why </b>

Duplicate records are one of the most common problems in data engineering pipelines.

Duplicates may appear due to:
- ingestion retries
- upstream system errors
- streaming reprocessing
- batch ingestion overlap

If not properly handled, duplicates can lead to:
- incorrect aggregations
- inflated metrics
- unreliable analytics
- incorrect business insights

Apache Spark provides multiple ways to handle duplicate records.
| Function                  | Purpose                                                     |
| ------------------------- | ----------------------------------------------------------- |
| `dropDuplicates()`        | remove duplicate rows optionally based on subset of columns |
| `distinct()`              | remove duplicate rows across the entire row                 |
| `groupBy().count()`       | detect duplicate records                                    |
| `approx_count_distinct()` | estimate cardinality efficiently                            |


<b> Important Difference </b> 

distinct() </br>
Removes duplicate rows considering all columns.

Equivalent to SQL:
```
SELECT DISTINCT * FROM table
```
Example:
```
df.distinct()
dropDuplicates()
```

Removes duplicates based on a subset of columns.

This is more flexible and commonly used in data pipelines.

Example:
```
df.dropDuplicates(["customer_id"])
```

<b> Key Concept </b>
| Method             | Behavior                                       |
| ------------------ | ---------------------------------------------- |
| `distinct()`       | removes duplicate rows considering all columns |
| `dropDuplicates()` | removes duplicates based on specific columns   |


<b> Your Task </b>

Perform deduplication across all Silver DataFrames.

1 - Customers

Remove duplicate customers based on:
- customer_id

Use:
- dropDuplicates(["customer_id"])

2 - Products

Remove duplicates based on:
- product_id

3 - Orders

Remove duplicates based on:
- order_id

4 - Order Items

Order items may contain duplicates due to ingestion retries.

Remove duplicates using the combination:
- order_id
- product_id

5 - Demonstrate distinct()

Use distinct() on the orders dataset to demonstrate how it behaves compared to dropDuplicates().

### Your solution

### Sugested solutions

In [0]:
# Before removing duplicates, we can detect them first.
from pyspark.sql.functions import count

duplicate_customers = (
    customers_silver_df
    .groupBy("customer_id")
    .agg(count("*").alias("records"))
    .filter(col("records") > 1)
)

display(duplicate_customers)

In [0]:
from pyspark.sql.functions import col

# -----------------------------------------------------
# DISTINCT example
# -----------------------------------------------------


orders_distinct_df = orders_silver_df


customers_silver_df = (
    customers_silver_df
    # dropDuplicates removes rows with duplicated values
    # based on a subset of columns
    .dropDuplicates(["customer_id"])
)

products_silver_df = (
    products_silver_df
    .dropDuplicates(["product_id"])
)

orders_silver_df = (
    orders_silver_df
    .dropDuplicates(["order_id"])
)

order_items_silver_df = (
    order_items_silver_df
    .dropDuplicates(["order_id", "product_id"])
)


# -----------------------------------------------------
# Compare record counts
# -----------------------------------------------------

print("Orders count:")
print(orders_distinct_df.count())

print("Orders count after dropDuplicates:")
print(orders_silver_df.count())

print("Orders count after distinct:")
# distinct() removes duplicate rows across ALL columns
print(orders_distinct_df.distinct().count())

# display(customers_silver_df)
# display(products_silver_df)
# display(orders_silver_df)
# display(order_items_silver_df)

## TASK 5 — Column Transformations and Data [Standardization](url)

<b> Why </b>

Column transformations are one of the most common operations in Spark pipelines.

At this stage of the Silver layer, we standardize and enrich the datasets to prepare them for analytical consumption in the Gold layer.

Typical transformations include:
- renaming columns
- casting data types
- conditional transformations
- date normalization
- timezone conversion
- handling null values
- deriving new attributes

These operations are frequently tested in the Spark certification from Databricks.

Key Functions Used
| Function          | Purpose                              |
| ----------------- | ------------------------------------ |
| `select()`        | select and reorder columns           |
| `selectExpr()`    | SQL expressions inside DataFrame API |
| `withColumn()`    | create or replace columns            |
| `alias()`         | rename columns                       |
| `cast()`          | convert data types                   |
| `when()`          | conditional logic                    |
| `coalesce()`      | fallback value for nulls             |
| `date_format()`   | format date values                   |
| `from_unixtime()` | convert unix timestamp to datetime   |


Your Task

Apply column transformations across all Silver DataFrames.

1️⃣ Customers — Standardize Columns</br>
- Perform the following transformations:
- rename customer_id → customer_key
- convert age to integer explicitly
- create age_group using when()
- ensure country has fallback using coalesce()
- convert created_at timestamp to Brazil timezone
- create a formatted date column using date_format()

2 -Products Transformations

Products may contain nested JSON attributes, so we avoid removing columns.

Transformations:

Transformation	Function
cast price	cast()
create price bucket	when()
create formatted timestamp	date_format()

3 - Orders — Timestamp Conversion </b>
Orders frequently contain timestamp fields that need timezone normalization.

We will:
- format dates
- convert timestamps
- standardize order status

4 - Order Items Transformations
Transformations include:

| Transformation       | Function       |
| -------------------- | -------------- |
| cast quantity        | `cast()`       |
| calculate item total | `withColumn()` |
| handle null price    | `coalesce()`   |


### Your solution

### Sugested solution

In [0]:
from pyspark.sql.functions import (
    col,
    lower,
    upper,
    when,
    coalesce,
    date_format
)
customers_silver_df = (
    customers_silver_df
    .select(
        "*",  # preserve all columns
        
        # standardize email format
        lower(col("email")).alias("email_standardized"),
        
        # normalize country values
        upper(col("country")).alias("country_standardized")
    )
    .withColumn(
        "age",
        col("age").cast("int")
    )
    .withColumn(
        "age_group",
        when(col("age") < 25, "young")
        .when(col("age") < 50, "adult")
        .otherwise("senior")
    )
    .withColumn(
        "country_standardized",
        coalesce(col("country_standardized"), col("country"))
    )
    .withColumn(
        "created_date",
        date_format(col("created_at"), "yyyy-MM-dd")
    )
)
customers_silver_df.limit(2).display()

In [0]:
# -----------------------------------------------------
# PRODUCTS — SQL Style Transformations
# -----------------------------------------------------
products_silver_df = (
    products_silver_df
    .withColumn(
        "price",
        col("price").cast("double")
    )
    .withColumn(
        "price_category",
        when(col("price") < 50, "low")
        .when(col("price") < 200, "medium")
        .otherwise("high")
    )
)

products_silver_df.limit(2).display()

In [0]:
from pyspark.sql.functions import to_timestamp

orders_silver_df = (
    orders_silver_df
    .selectExpr(
        "*",
        
        # SQL expression example
        "upper(status) as order_status_standardized"
    )
    .withColumn(
        "order_timestamp",
        to_timestamp(col("order_timestamp"))
    )
    .withColumn(
        "order_day",
        date_format(col("order_timestamp"), "yyyy-MM-dd")
    )
)

orders_silver_df.limit(2).display()

In [0]:
order_items_silver_df = (
    order_items_silver_df
    .withColumn(
        "quantity",
        col("quantity").cast("int")
    )
    .withColumn(
        "price",
        col("price").cast("double")
    )
    .withColumn(
        "price",
        coalesce(col("price"), col("price"))
    )
    .withColumn(
        "item_total",
        col("quantity") * col("price")
    )
)

order_items_silver_df.limit(2).display()

### Another exemples 

Example Using selectExpr()

selectExpr() allows using SQL expressions directly inside the DataFrame API.

Example:

In [0]:
orders_expr_df = orders_silver_df.selectExpr(
    "*",
    "year(order_timestamp) as order_year",
    "month(order_timestamp) as order_month"
)
orders_expr_df.limit(5).display()


Unix Timestamp Example

If a dataset contains Unix timestamps, we can convert them using:

In [0]:
from pyspark.sql.functions import from_unixtime

orders_silver_df_unix = orders_silver_df.withColumn(
    "order_timestamp_readable",
    from_unixtime(col("order_timestamp"))
)

# An error will be occur, because we dont have a unix column
# orders_silver_df_unix.limit(5).display()

Brazil Timezone Normalization (São Paulo)

When processing global datasets, timestamps often need timezone adjustments.

In [0]:
from pyspark.sql.functions import from_utc_timestamp

orders_silver_df_sp = orders_silver_df.withColumn(
    "order_time_brazil",
    from_utc_timestamp(col("order_timestamp"), "America/Sao_Paulo")
)

orders_silver_df_sp.limit(2).display()

## TASK 6 — Nested JSON Processing

<b> Why </b>

Modern data pipelines frequently ingest semi-structured data, such as JSON documents coming from:
- APIs
- event streams
- microservices
- NoSQL databases

These datasets commonly contain nested structures, including:
- structs
- arrays
- maps
- arrays of structs

Apache Spark provides powerful functions to transform and normalize nested data into tabular structures.

In the Silver layer we typically:
- extract nested attributes
- flatten complex structures
- enrich nested objects
- prepare datasets for analytics

Input Nested Schema (Products)

Your products_bronze_df contains the following nested structure:
```
product
 ├── attributes (struct)
 │    ├── color
 │    └── size
 │
 ├── tags (array<string>)
 │
 └── variants (array<struct>)
      ├── variant_id
      └── stock
```
Key Functions Used

| Function              | Purpose               |
| --------------------- | --------------------- |
| `flatten()`           | flatten nested arrays |
| `concat()`            | concat elements       |
| `map_concat()`        | merge maps            |
| `withColumn()`        | modify nested fields  |
| `struct()`            | create struct columns |
| `col("struct.field")` | access struct fields  |



<b> Your Task </b>

Perform the following transformations on products_silver_df.

1 - Extract fields from struct

Extract the following fields:
- attributes.color
- attributes.size

Create new columns:
- product_color
- product_size

2 - Rename fields inside struct </br> 
Create a new struct called:
- product_attributes

Renaming:
```
color → product_color
size → product_size
```

3 - Merge arrays

Merge the following arrays:
- tags

with a new array containing:
```
["spark", "databricks"]
```

Using:
- concat()

4 - Flatten nested arrays

Create a nested array and flatten it using:
- flatten()

5 - Create and merge maps

Create two maps and merge them using:
- map_concat()

### Your Solution

### Sugested solution

In [0]:
from pyspark.sql.functions import col, struct, array, flatten, create_map, map_concat, lit

# ---------------------------------------------------------
# Extract struct fields
# ---------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_color",
    col("attributes.color")
).withColumn(
    "product_size",
    col("attributes.size")
)

# ---------------------------------------------------------
# Rename fields inside struct
# ---------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "product_attributes",
    struct(
        col("attributes.color").alias("product_color"),
        col("attributes.size").alias("product_size")
    )
)

# ---------------------------------------------------------
# Merge arrays using concat
# ---------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "tags_enriched",
    concat(
        col("tags"),
        array(lit("spark"), lit("databricks"))
    )
)

# ---------------------------------------------------------
# Create nested array then flatten
# ---------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "nested_tags",
    array(col("tags"), col("tags_enriched"))
)

products_silver_df = products_silver_df.withColumn(
    "flattened_tags",
    flatten(col("nested_tags"))
)

# ---------------------------------------------------------
# Create maps and merge them
# ---------------------------------------------------------

products_silver_df = products_silver_df.withColumn(
    "map_1",
    create_map(
        lit("source"), lit("generator"),
        lit("layer"), lit("silver")
    )
)

products_silver_df = products_silver_df.withColumn(
    "map_2",
    create_map(
        lit("framework"), lit("spark"),
        lit("platform"), lit("databricks")
    )
)

products_silver_df = products_silver_df.withColumn(
    "metadata_map",
    map_concat(col("map_1"), col("map_2"))
)

display(products_silver_df)

## TASK 7 — Join Datasets

<b> Why </b>

Data pipelines frequently require combining multiple datasets.

Spark supports multiple join strategies including:
| Join Strategy     | Description                      |
| ----------------- | -------------------------------- |
| Broadcast Join    | small table broadcast to workers |
| Sort Merge Join   | default large dataset join       |
| Shuffle Hash Join | alternative shuffle join         |

<b> Why </b>

Data pipelines rarely operate on a single dataset.

Most analytical workloads require combining multiple datasets to produce enriched views of the data.

Examples include:
- joining orders with customers
j- oining order items with products
generating sales datasets

Apache Spark supports multiple join strategies, depending on dataset size and distribution.

In this task we will build enriched datasets that will later be used in the Gold analytical layer.

Join Relationships

The datasets in this project are related as follows:
```
customers
    │
    │ customer_id
    ▼
orders
    │
    │ order_id
    ▼
order_items
    │
    │ product_id
    ▼
products
```

Key Functions Used
| Function       | Purpose                         |
| -------------- | ------------------------------- |
| `join()`       | combine datasets                |
| `broadcast()`  | optimize joins for small tables |
| `select()`     | select output columns           |
| `alias()`      | rename columns                  |
| `withColumn()` | create derived columns          |


<b> Your Task </b> 

Create enriched datasets using joins.

1 - Join Orders with Customers

Create an enriched dataset containing:
- order information
- customer attributes

Join key:
- customer_id

Join type:
- left join

Reason:

We want to keep all orders, even if customer data is missing.


Why Broadcast?

customers is typically a small dimension table.

Broadcasting it avoids a large shuffle.

2 - Join Order Items with Products

Enrich order items with product information.

Join key: product_id

3 - Create Full Sales Dataset

Now combine everything:
```
orders + order_items + products + customers
```
This creates a complete transactional dataset.

4 - Create Derived Metrics

Add useful metrics for analytics.

### Your solution

### Sugested solution

In [0]:
from pyspark.sql.functions import broadcast

orders_customers_df = (
    orders_silver_df
    .join(
        broadcast(customers_silver_df),  # optimize join
        "customer_id",
        "left"
    )
)

orders_customers_df.limit(2).display()

In [0]:
order_items_products_df = (
    order_items_silver_df
    .join(
        broadcast(products_silver_df.withColumnRenamed("price", 'products_price')),
        "product_id",
        "left"
    )
)

order_items_products_df.limit(2).display()

In [0]:

order_customer_price = orders_customers_df.withColumn("order_customer_price", col("price")).drop("price")
sales_enriched_df = (
    order_items_products_df
    .join(
        orders_customers_df,
        "order_id",
        "left"
    )
)

sales_enriched_df.limit(2).display()

In [0]:
%python
from pyspark.sql.functions import col
sales_enriched_df = sales_enriched_df.withColumn(
    "total_item_value",
    col("quantity") * col("price")
)


sales_enriched_df.limit(5).display()

### Important

<b> Join Types in Spark </b>
Spark supports several join types:
| Join Type | Description                   |
| --------- | ----------------------------- |
| inner     | rows that match both datasets |
| left      | all rows from left dataset    |
| right     | all rows from right dataset   |
| full      | all rows from both datasets   |
| left_semi | filter rows that match        |
| left_anti | filter rows that do not match |

Inspecting Join Strategy
Spark can show the execution plan.
```
sales_enriched_df.explain(True)
```
This prints:
- logical plan
- optimized plan
- physical plan

The physical plan may show:
- BroadcastHashJoin
- SortMergeJoin
- ShuffleHashJoin

In [0]:
sales_enriched_df.explain(True)

## TASK 8 — Partition Optimization and Write Tables

Why

Efficient partitioning is critical for performance and scalability in Apache Spark pipelines.

Partitioning affects:
- parallel processing
- shuffle operations
- file size optimization
- query performance

Before writing datasets to storage, we can adjust the number and structure of partitions.

Spark provides three key tools for this purpose.

Key Functions Used
| Function        | Purpose                                          |
| --------------- | ------------------------------------------------ |
| `repartition()` | increase or rebalance partitions with shuffle    |
| `coalesce()`    | reduce number of partitions without full shuffle |
| `partitionBy()` | physically partition output data when writing    |

repartition() vs coalesce()
| Function        | Behavior                                          |
| --------------- | ------------------------------------------------- |
| `repartition()` | performs full shuffle to evenly redistribute data |
| `coalesce()`    | reduces partitions without full shuffle           |


Example:
- df.repartition(10)

Redistributes data across 10 partitions.

Example:

- df.coalesce(2)

Reduces partitions to 2, usually faster.


Why Partition Data When Writing </br>
Partitioning storage improves query performance.

Example:
```
sales/
 ├── country=US/
 ├── country=BR/
 ├── country=CA/
```
When filtering:
```
WHERE country = 'BR'
```

Spark only scans that partition.

Your Task

Perform partition optimization before writing Silver tables.

1 -Repartition the datasets

Apply repartition() based on relevant columns.

| Dataset     | Partition Column |
| ----------- | ---------------- |
| customers   | country          |
| products    | category         |
| orders      | order_day        |
| order_items | order_id         |

2 - Reduce partitions before writing

Use coalesce() to reduce the number of output files.

3 - Save Silver tables using partitionBy()

Write the final datasets as Delta tables.

### Write Modes in Apache Spark

When writing DataFrames to storage, Spark provides several write modes that control what happens if the destination data already exists.

These modes are used with:
- df.write.mode("mode_name")

Write Modes Overview
| Mode                      | Behavior                       | Typical Use Case     |
| ------------------------- | ------------------------------ | -------------------- |
| `overwrite`               | replaces existing data         | rebuilding tables    |
| `append`                  | adds new rows to existing data | incremental loads    |
| `ignore`                  | skips write if data exists     | idempotent pipelines |
| `error` / `errorifexists` | throws an error if data exists | safety validation    |

Write Mode Comparison
| Mode      | Overwrites Data | Adds Data | Throws Error |
| --------- | --------------- | --------- | ------------ |
| overwrite | ✅               | ❌         | ❌            |
| append    | ❌               | ✅         | ❌            |
| ignore    | ❌               | ❌         | ❌            |
| error     | ❌               | ❌         | ✅            |


Exemples:
With partitionBy:
```
df.write \
  .mode("append") \
  .partitionBy("country") \
  .format("delta") \
  .saveAsTable("sales")
```

In external Location File
```
df.write \
  .mode("append") \
  .partitionBy("country") \
  .format("delta") \
  .option("path", "/Volumes/my_catalog/my_schema/my_volume/sales") \
  .saveAsTable("sales")
```

Save x SaveAsTable
| Método                           | Cria tabela | Define path |
| -------------------------------- | ----------- | ----------- |
| `save()`                         | ❌           | ✅           |
| `saveAsTable()`                  | ✅           | ❌           |
| `saveAsTable() + option("path")` | ✅           | ✅           |



### Your Solution

### Sugested solution

In [0]:
customers_silver_df = customers_silver_df.repartition("country_standardized")
customers_silver_df = customers_silver_df.coalesce(4)

customers_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("country_standardized") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_SILVER}.customers")

In [0]:
products_silver_df = products_silver_df.repartition("category")
products_silver_df = products_silver_df.coalesce(4)
products_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("category") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_SILVER}.products")

In [0]:
orders_silver_df = orders_silver_df.repartition("order_day")
orders_silver_df = orders_silver_df.coalesce(4)
orders_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_day") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_SILVER}.orders")

In [0]:
order_items_silver_df = order_items_silver_df.repartition("order_id")
order_items_silver_df = order_items_silver_df.coalesce(4)
order_items_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_id") \
    .saveAsTable(f"{CATALOG_NAME}.{SCHEMA_SILVER}.order_items")

### Checking Partition Distribution

You can inspect the number of partitions:

In [0]:
# customers_silver_df.rdd.getNumPartitions()